In [1]:
from pathlib import Path
import re
import unicodedata
import pandas as pd


# =========================
# 1. Settings
# =========================

data_dir = Path("./original_data/cheonan_pop")
out_path = Path("./pop_by_age_201901_202605.csv")

start_ym = "201901"
end_ym = "202605"

file_pattern = "*연령별인구현황_월간.csv"
keep_dong_only = True


# =========================
# 2. Column definitions
# =========================

AGE_COLS = [f"age_{i}" for i in range(0, 101)]
MALE_AGE_COLS = [f"male_{i}" for i in range(0, 101)]
FEMALE_AGE_COLS = [f"female_{i}" for i in range(0, 101)]

VALUE_COLS = (
    ["total_pop", "male_pop", "female_pop"]
    + AGE_COLS
    + MALE_AGE_COLS
    + FEMALE_AGE_COLS
)

FINAL_COLS = (
    ["adm_cd", "district", "adm_nm", "date", "ym"]
    + VALUE_COLS
)


# =========================
# 3. Helper functions
# =========================

def read_csv_safe(path):
    encodings = ["utf-8-sig", "cp949", "euc-kr"]

    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc, dtype=str)
        except UnicodeDecodeError:
            continue

    raise ValueError(f"Cannot read file: {path}")


def normalize_text(x):
    x = str(x)
    x = unicodedata.normalize("NFKC", x)
    x = x.replace("\ufeff", "")
    x = x.replace("\u200b", "")
    x = x.replace("\xa0", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()


def split_area(area):
    area = normalize_text(area)

    code_match = re.search(r"\((\d+)\)$", area)
    adm_cd = code_match.group(1) if code_match else pd.NA

    name = re.sub(r"\s*\(\d+\)$", "", area).strip()
    name = name.replace("충청남도 천안시", "").strip()

    if name == "":
        return pd.Series([adm_cd, pd.NA, "천안시"])

    parts = name.split()

    if len(parts) >= 2 and parts[0] in ["동남구", "서북구"]:
        district = parts[0]
        adm_nm = parts[-1]
    elif len(parts) == 1 and parts[0] in ["동남구", "서북구"]:
        district = parts[0]
        adm_nm = parts[0]
    else:
        district = pd.NA
        adm_nm = parts[-1] if parts else name

    return pd.Series([adm_cd, district, adm_nm])


def clean_num_series(s):
    return (
        s.astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
            "<NA>": pd.NA
        })
        .pipe(pd.to_numeric, errors="coerce")
        .astype("Int64")
    )


def parse_column(col):
    col = normalize_text(col)

    m = re.fullmatch(r"(\d{4})년\s*(\d{1,2})월_(계|남|여)_(.+)", col)

    if not m:
        return None

    year, month, sex, item = m.groups()
    ym = f"{year}{int(month):02d}"
    item = normalize_text(item)

    if not (start_ym <= ym <= end_ym):
        return None

    if item == "연령구간인구수":
        return None

    if item == "총인구수":
        var_map = {
            "계": "total_pop",
            "남": "male_pop",
            "여": "female_pop"
        }
        return ym, var_map[sex]

    age_match = re.search(r"\d+", item)

    if age_match is None:
        return None

    age = int(age_match.group())

    if age < 0 or age > 100:
        return None

    prefix_map = {
        "계": "age",
        "남": "male",
        "여": "female"
    }

    var = f"{prefix_map[sex]}_{age}"

    return ym, var


def reshape_file(path):
    raw = read_csv_safe(path)
    raw.columns = [normalize_text(c) for c in raw.columns]

    if "행정구역" not in raw.columns:
        raise ValueError(f"{path.name}: '행정구역' column not found.")

    area = raw[["행정구역"]].copy()
    area[["adm_cd", "district", "adm_nm"]] = area["행정구역"].apply(split_area)
    area = area.drop(columns=["행정구역"])

    raw = pd.concat([area, raw.drop(columns=["행정구역"])], axis=1)

    value_cols = []
    col_info = {}

    for col in raw.columns:
        parsed = parse_column(col)

        if parsed is None:
            continue

        ym, var = parsed
        value_cols.append(col)
        col_info[col] = (ym, var)

    if not value_cols:
        return pd.DataFrame(columns=FINAL_COLS)

    long_df = raw.melt(
        id_vars=["adm_cd", "district", "adm_nm"],
        value_vars=value_cols,
        var_name="raw_col",
        value_name="pop"
    )

    long_df["ym"] = long_df["raw_col"].map(lambda x: col_info[x][0])
    long_df["var"] = long_df["raw_col"].map(lambda x: col_info[x][1])

    long_df["pop"] = clean_num_series(long_df["pop"])

    wide = (
        long_df
        .pivot_table(
            index=["adm_cd", "district", "adm_nm", "ym"],
            columns="var",
            values="pop",
            aggfunc="first"
        )
        .reset_index()
    )

    wide.columns.name = None

    wide["date"] = pd.to_datetime(wide["ym"] + "01", format="%Y%m%d")

    for col in VALUE_COLS:
        if col not in wide.columns:
            wide[col] = pd.NA

    wide = wide[FINAL_COLS]

    print(
        f"{path.name}: "
        f"{wide['ym'].min()} ~ {wide['ym'].max()}, "
        f"rows={len(wide):,}"
    )

    return wide


# =========================
# 4. Merge files
# =========================

files = sorted(data_dir.glob(file_pattern))

if not files:
    raise FileNotFoundError(
        f"No files found in {data_dir.resolve()} with pattern: {file_pattern}"
    )

print("Current directory:", data_dir.resolve())
print("Input files:")
for f in files:
    print(" -", f.name)

frames = []

for path in files:
    temp = reshape_file(path)

    if not temp.empty:
        frames.append(temp)

if not frames:
    raise ValueError("No valid data was loaded.")

pop = pd.concat(frames, ignore_index=True)


# =========================
# 5. Keep eup/myeon/dong only
# =========================

if keep_dong_only:
    pop = pop[
        pop["district"].isin(["동남구", "서북구"])
        & pop["adm_nm"].str.contains(r"(?:읍|면|동)$", regex=True, na=False)
    ].copy()


# =========================
# 5-1. Remove zero-population rows
# =========================

pop["total_pop"] = pd.to_numeric(pop["total_pop"], errors="coerce").astype("Int64")

zero_rows = pop[pop["total_pop"] == 0].copy()

if not zero_rows.empty:
    print("\nRemoved zero-population rows:")
    print(
        zero_rows[
            ["ym", "district", "adm_nm", "total_pop"]
        ].sort_values(["ym", "district", "adm_nm"]).to_string(index=False)
    )

pop = pop[pop["total_pop"] > 0].copy()


# =========================
# 6. Remove duplicates and sort
# =========================

pop = pop.drop_duplicates(
    subset=["adm_cd", "district", "adm_nm", "ym"],
    keep="last"
)

pop = pop.sort_values(
    ["district", "adm_nm", "ym"],
    ignore_index=True
)


# =========================
# 7. Validation
# =========================

for col in VALUE_COLS:
    pop[col] = pd.to_numeric(pop[col], errors="coerce").astype("Int64")

print("\nRows by month:")
print(pop["ym"].value_counts().sort_index())

check = pop.copy()

check["age_sum"] = check[AGE_COLS].sum(axis=1)
check["male_sum"] = check[MALE_AGE_COLS].sum(axis=1)
check["female_sum"] = check[FEMALE_AGE_COLS].sum(axis=1)

bad_total = check[check["age_sum"] != check["total_pop"]]
bad_male = check[check["male_sum"] != check["male_pop"]]
bad_female = check[check["female_sum"] != check["female_pop"]]

if not bad_total.empty or not bad_male.empty or not bad_female.empty:
    print("Population sum validation failed.")
    print("bad_total:", len(bad_total))
    print("bad_male:", len(bad_male))
    print("bad_female:", len(bad_female))

    print("\nSample failed rows:")
    print(
        check.loc[
            bad_total.index[:5],
            ["district", "adm_nm", "ym", "total_pop", "age_sum"]
        ]
    )

    raise ValueError("Population sum validation failed.")


# =========================
# 8. Save
# =========================

if out_path.exists():
    out_path.unlink()

pop.to_csv(out_path, index=False, encoding="utf-8-sig")

print("\nDone.")
print(f"Files merged: {len(files)}")
print(f"Rows: {len(pop):,}")
print(f"Columns: {len(pop.columns):,}")
print(f"Saved to: {out_path.resolve()}")

Current directory: C:\Users\심현석\Documents\GitHub\Cheonan-Data-Analysis\Hyeonseok\data\cheonan_pop
Input files:
 - 201901_201903_연령별인구현황_월간.csv
 - 201904_201906_연령별인구현황_월간.csv
 - 201907_201909_연령별인구현황_월간.csv
 - 201910_201912_연령별인구현황_월간.csv
 - 202001_202003_연령별인구현황_월간.csv
 - 202004_202006_연령별인구현황_월간.csv
 - 202007_202009_연령별인구현황_월간.csv
 - 202010_202012_연령별인구현황_월간.csv
 - 202101_202103_연령별인구현황_월간.csv
 - 202104_202106_연령별인구현황_월간.csv
 - 202107_202109_연령별인구현황_월간.csv
 - 202110_202112_연령별인구현황_월간.csv
 - 202201_202203_연령별인구현황_월간.csv
 - 202204_202206_연령별인구현황_월간.csv
 - 202207_202209_연령별인구현황_월간.csv
 - 202210_202212_연령별인구현황_월간.csv
 - 202301_202303_연령별인구현황_월간.csv
 - 202304_202306_연령별인구현황_월간.csv
 - 202307_202309_연령별인구현황_월간.csv
 - 202310_202312_연령별인구현황_월간.csv
 - 202401_202403_연령별인구현황_월간.csv
 - 202404_202406_연령별인구현황_월간.csv
 - 202407_202409_연령별인구현황_월간.csv
 - 202410_202412_연령별인구현황_월간.csv
 - 202501_202503_연령별인구현황_월간.csv
 - 202504_202506_연령별인구현황_월간.csv
 - 202507_202509_연령별인구현황_월간.csv
 - 202510_202512_연령별인구현황_